In [2]:
import numpy as np
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import LabelEncoder

In [3]:
df = pd.read_csv('https://raw.githubusercontent.com/gscdit/Breast-Cancer-Detection/refs/heads/master/data.csv')
df.head()

,id,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,...,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst,Unnamed: 32
0,842302,M,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,NaN
1,842517,M,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,NaN
2,84300903,M,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,NaN
3,84348301,M,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,NaN
4,84358402,M,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,NaN


In [4]:
df.columns

Index(['id', 'diagnosis', 'radius_mean', 'texture_mean', 'perimeter_mean',
       'area_mean', 'smoothness_mean', 'compactness_mean', 'concavity_mean',
       'concave points_mean', 'symmetry_mean', 'fractal_dimension_mean',
       'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
       'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se',
       'fractal_dimension_se', 'radius_worst', 'texture_worst',
       'perimeter_worst', 'area_worst', 'smoothness_worst',
       'compactness_worst', 'concavity_worst', 'concave points_worst',
       'symmetry_worst', 'fractal_dimension_worst', 'Unnamed: 32'],
      dtype='object')

In [5]:
df.shape

(569, 33)

In [6]:
df.drop(columns=["id", "Unnamed: 32"], inplace = True)

In [7]:
df.shape

(569, 31)

In [8]:
X_train, X_test, y_train, y_test = train_test_split(df.iloc[:, 1:], df.iloc[:, 0], test_size=0.2)

In [9]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [10]:
X_train

array([[ 1.12511958,  1.18023248,  0.99979754, ..., -0.02030788,
        -0.66101069, -0.2605597 ],
       [ 1.21802035,  0.14134457,  1.18521365, ...,  1.1939852 ,
         0.26627237, -0.03493878],
       [-0.10581564, -0.76133359, -0.15315352, ..., -0.68740632,
         0.69756682, -0.39324629],
       ...,
       [-0.37871165, -0.61588929, -0.37775985, ..., -0.52600908,
         0.27954297,  0.06659064],
       [ 0.2454654 ,  0.73004772,  0.44776323, ...,  2.02232834,
        -0.06051612,  1.67440831],
       [ 2.6812075 ,  1.68582459,  2.8413166 , ...,  2.41590574,
         0.49021372,  0.17671514]])

In [11]:
encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [12]:
X_train = torch.tensor(X_train,dtype=torch.float32)
X_test = torch.tensor(X_test,dtype=torch.float32)
y_train = torch.tensor(y_train,dtype=torch.float32)
y_test = torch.tensor(y_test,dtype=torch.float32)

In [13]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

(torch.Size([455, 30]),
 torch.Size([114, 30]),
 torch.Size([455]),
 torch.Size([114]))

In [14]:
class MysimpleNN():
  def __init__(self,x):
    self.weights = torch.rand(x.shape[1],1,dtype=torch.float32,requires_grad=True)
    self.bias = torch.rand(1,dtype=torch.float32,requires_grad=True)

  def forward(self,x):
    z = torch.matmul(x,self.weights) + self.bias
    y_pred = torch.sigmoid(z)
    return y_pred

  def loss_function(self,y_pred,y):
    epsilon = 1e-7
    y_pred = torch.clamp(y_pred, epsilon, 1-epsilon)

    loss = -(y_train * torch.log(y_pred) + (1 - y_train) * torch.log(1 - y_pred))
    return loss.mean()

In [15]:
import torch.nn as nn

In [19]:
class MysimpleNNusingNN(nn.Module):
  def __init__(self,num_features):

    super().__init__()
    self.network = nn.Sequential(
        nn.Linear(num_features, 3),
        nn.ReLU(),
        nn.Linear(3, 1),
        nn.Sigmoid()
    )

  def forward(self,features):

    out = self.network(features)

    return out

In [20]:
lr = 0.1
epochs = 25

loss_function = nn.BCELoss()

In [23]:
model = MysimpleNNusingNN(num_features=X_train.shape[1])

optimizer = torch.optim.SGD(model.parameters(),lr = lr)


for epoch in range(epochs):
  y_pred = model(X_train)

  loss= loss_function(y_pred,y_train.view(-1,1))

  optimizer.zero_grad()

  loss.backward()

  optimizer.step()



  print(f"Epoch : {epoch+1}, Loss: {loss.item()} ")


Epoch : 1, Loss: 0.6543159484863281 
Epoch : 2, Loss: 0.6247534155845642 
Epoch : 3, Loss: 0.5980969071388245 
Epoch : 4, Loss: 0.5739285945892334 
Epoch : 5, Loss: 0.5520414710044861 
Epoch : 6, Loss: 0.5312966108322144 
Epoch : 7, Loss: 0.5116018056869507 
Epoch : 8, Loss: 0.4928828775882721 
Epoch : 9, Loss: 0.4751255512237549 
Epoch : 10, Loss: 0.45820873975753784 
Epoch : 11, Loss: 0.4423101842403412 
Epoch : 12, Loss: 0.4271681010723114 
Epoch : 13, Loss: 0.4128030836582184 
Epoch : 14, Loss: 0.39914312958717346 
Epoch : 15, Loss: 0.3861728310585022 
Epoch : 16, Loss: 0.37389153242111206 
Epoch : 17, Loss: 0.362296998500824 
Epoch : 18, Loss: 0.35133522748947144 
Epoch : 19, Loss: 0.34099113941192627 
Epoch : 20, Loss: 0.33123019337654114 
Epoch : 21, Loss: 0.32198044657707214 
Epoch : 22, Loss: 0.31324782967567444 
Epoch : 23, Loss: 0.3049819767475128 
Epoch : 24, Loss: 0.2971815764904022 
Epoch : 25, Loss: 0.2898156940937042 


In [26]:
model = MysimpleNN(x=X_train)

for epoch in range(epochs):
  y_pred = model.forward(X_train)

  loss= model.loss_function(y_pred,y_train)

  loss.backward()

  with torch.no_grad():
    model.weights -= lr * model.weights.grad
    model.bias -= lr * model.bias.grad

  model.weights.grad.zero_()
  model.bias.grad.zero_()

  print(f"Epoch : {epoch+1}, Loss: {loss.item()} ")


Epoch : 1, Loss: 3.476132392883301 
Epoch : 2, Loss: 3.343181610107422 
Epoch : 3, Loss: 3.2070274353027344 
Epoch : 4, Loss: 3.0705103874206543 
Epoch : 5, Loss: 2.9340007305145264 
Epoch : 6, Loss: 2.794062376022339 
Epoch : 7, Loss: 2.6474411487579346 
Epoch : 8, Loss: 2.501413583755493 
Epoch : 9, Loss: 2.3602328300476074 
Epoch : 10, Loss: 2.220597505569458 
Epoch : 11, Loss: 2.078655481338501 
Epoch : 12, Loss: 1.9410537481307983 
Epoch : 13, Loss: 1.806514024734497 
Epoch : 14, Loss: 1.6779991388320923 
Epoch : 15, Loss: 1.5579410791397095 
Epoch : 16, Loss: 1.4494457244873047 
Epoch : 17, Loss: 1.353745937347412 
Epoch : 18, Loss: 1.2708015441894531 
Epoch : 19, Loss: 1.2002224922180176 
Epoch : 20, Loss: 1.1385396718978882 
Epoch : 21, Loss: 1.0875270366668701 
Epoch : 22, Loss: 1.0450656414031982 
Epoch : 23, Loss: 1.0094321966171265 
Epoch : 24, Loss: 0.9790714979171753 
Epoch : 25, Loss: 0.9528403878211975 


In [27]:
model.bias

tensor([0.3369], requires_grad=True)

In [21]:
from sklearn.metrics import accuracy_score

In [24]:
with torch.no_grad():
  y_pred = model.forward(X_test)
  print(accuracy_score(y_test,y_pred.round()))

0.9736842105263158


In [25]:
# model evaluation
with torch.no_grad():
  y_pred = model.forward(X_test)
  y_pred = (y_pred > 0.9).float()
  accuracy = (y_pred == y_test).float().mean()
  print(f'Accuracy: {accuracy.item()}')
  print(f'Accuracy: {accuracy}')


Accuracy: 0.6046475768089294
Accuracy: 0.6046475768089294
